
## Window Functions and Analytical Functions:

- A Window Function in PySpark is used to `performs calculations over a group of rows without combining them into a single row`.
- It allows us to `calculate rankings, running totals, averages, and comparisons`.
- First can define `a windows specification(partitionBy(), orderBy(), rows_between(), range_between())`, then only can define `windows functions(rank(), dense_rank(), row_number(), lag(), lead())`.

In [0]:
%run ./04_GroupBy_Aggregations


## Windows Functions Starts here


### 1. row_number():

**Assign a unique sequential number to each customer's orders.**

In [0]:
customer_window = Window.partitionBy( "customer_id").orderBy(
    F.col("order_date").asc(),
    F.col("order_id").asc()
)

customer_orders_ranked_df = orders_clean_df.withColumn(
    "order_number",
    F.row_number().over(customer_window)
)

display(customer_orders_ranked_df)


**Business use: Identify each customer's first, second, and subsequent orders.**


### 2. rank() and dense_rank():

**Rank products by total revenue within each category.**

In [0]:
product_revenue_df = sales_df.groupBy(
    "category",
    "product_id",
    "product_name"
).agg(
    F.sum("net_revenue").alias("total_revenue")
)

product_window = Window.partitionBy(  "category" ).orderBy(
    F.col("total_revenue").desc()
)

product_ranked_df = (
    product_revenue_df
    .withColumn(
        "row_number",
        F.row_number().over(product_window)
    )
    .withColumn(
        "rank",
        F.rank().over(product_window)
    )
    .withColumn(
        "dense_rank",
        F.dense_rank().over(product_window)
    )
)

display(product_ranked_df)


### 3. lag():

**Find the previous order date for each customer.**

In [0]:
customer_order_window = Window.partitionBy( "customer_id" ).orderBy(
    "order_date",
    "order_id"
)

customer_order_history_df = orders_clean_df.withColumn(
    "previous_order_date",
    F.lag("order_date", 1).over(customer_order_window)
)

display(customer_order_history_df)


**Business use: Measure how much time passes between a customer's purchases.**


### 4. lead():

**Find the next order date for each customer.**

In [0]:
customer_order_window = Window.partitionBy( "customer_id" ).orderBy(
    "order_date",
    "order_id"
)

customer_order_history_df = customer_order_history_df.withColumn(
    "next_order_date",
    F.lead("order_date", 1).over(customer_order_window)
)

display(customer_order_history_df)


**Business use: Identify the next purchase after an order.**


### 5. Running Total:

**Calculate cumulative revenue for each customer.**

In [0]:
running_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date", "order_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

customer_running_revenue_df = sales_df.withColumn(
    "running_revenue",
    F.sum("net_revenue").over(running_window)
)

display(customer_running_revenue_df)


**Business use: Track how much each customer has spent up to each order item.**


### 6. Compare Current Revenue with Previous Month:

In [0]:
monthly_window = Window.orderBy("order_month")

monthly_comparison_df = monthly_sales_df.withColumn(
    "previous_month_revenue",
    F.lag("monthly_revenue", 1).over(monthly_window)
).withColumn(
    "revenue_change",
    F.col("monthly_revenue") - F.col("previous_month_revenue")
)

display(monthly_comparison_df)


**Business use: Identify whether monthly revenue increased or decreased.**